# 01 · Collect and parse

Downloads the pinned sources to `data/raw/` (unchanged, logged in `data/raw/manifest.csv`)
and parses them into long tables in `data/interim/`, one row per poll × scenario × candidate.

Scope: collection and parsing only. No validation analysis, thinning of rolling polls or statistics.

* **Part 1 (network).** Resolves any unpinned source in `config/sources.yaml` and downloads what isn't cached yet.
* **Part 2 (offline).** Reads only `data/raw/`. Run it alone to re-parse without a connection.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from polls import fetch, wikipedia, france_politique, nsppolls, results, candidates, institutes, assemble, checks

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

SOURCES = ROOT / "config" / "sources.yaml"
CANDIDATES = ROOT / "config" / "candidates.csv"
INSTITUTES = ROOT / "config" / "institutes.csv"
RAW = ROOT / "data" / "raw"
MANIFEST = RAW / "manifest.csv"
INTERIM = ROOT / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)

## Part 1 · Collection (network)

In [2]:
cfg = fetch.load_sources(SOURCES)
http = fetch.PoliteSession()
newly_pinned = fetch.resolve_pins(cfg, http)
print("Newly pinned:", newly_pinned or "nothing (all sources already pinned)")

Newly pinned: nothing (all sources already pinned)


In [3]:
fetch.fetch_wikipedia(cfg, http, RAW, MANIFEST, ROOT)
fetch.fetch_france_politique(cfg, http, RAW, MANIFEST, ROOT)
fetch.fetch_nsppolls(cfg, http, RAW, MANIFEST, ROOT)
fetch.fetch_results(cfg, http, RAW, MANIFEST, ROOT)
fetch.save_sources(cfg, SOURCES)   # records any new pin, including a new Wayback capture

pd.read_csv(MANIFEST)

,source,url,pinned_id,retrieved_utc,sha256,local_path,archive_url
0,wikipedia_2002,https://fr.wikipedia.org/w/api.php?action=parse&oldid=23...,236399441,2026-10-01T09:34:37+00:00,95ac392f1136308308b58e7b9440c21e9713291cc80dace8c1fbb627...,data/raw/wikipedia/2002_rev236399441.json,NaN
1,wikipedia_2007,https://fr.wikipedia.org/w/api.php?action=parse&oldid=23...,233109259,2026-10-01T09:34:39+00:00,4f8b7b5ed756436d3bbfde49e03411fad2a8a01992aaf525b50c3133...,data/raw/wikipedia/2007_rev233109259.json,NaN
2,wikipedia_2012,https://fr.wikipedia.org/w/api.php?action=parse&oldid=23...,238741176,2026-10-01T09:34:41+00:00,8cd5bfe60a97e9e141a1b11194ef9ee6f2ba4d9ede56b25f2db9659a...,data/raw/wikipedia/2012_rev238741176.json,NaN
3,wikipedia_2017,https://fr.wikipedia.org/w/api.php?action=parse&oldid=23...,239719711,2026-10-01T09:34:44+00:00,a1d183955a0956053d9575f43638a27b80979303549bcfe3fd713150...,data/raw/wikipedia/2017_rev239719711.json,NaN
4,wikipedia_2022,https://fr.wikipedia.org/w/api.php?action=parse&oldid=23...,239942428,2026-10-01T09:34:46+00:00,9e78f8759f88575d2124a452cf17d59544b62abc23851b02074d8ee6...,data/raw/wikipedia/2022_rev239942428.json,NaN
5,nsppolls_2022,https://raw.githubusercontent.com/nsppolls/nsppolls/ddcd...,ddcd5ceef9839c7d3f46948ff85f41fc93f215ce,2026-10-01T09:34:48+00:00,2dabc16a43a730a90616a12725585b73faaa2b1bac614e5c16a34010...,data/raw/nsppolls/ddcd5ceef983_presidentielle.json,NaN
6,results,https://raw.githubusercontent.com/clarasalas/strategic-v...,f666a8b36e8c93f22d7f3373cf4f8704648ecfb4,2026-10-01T09:34:50+00:00,cc1f9d49dc60cd87323ae78437b3b0080a51bdada44bcb9cbae6bbe8...,data/raw/results/f666a8b36e8c_FR-electoral_data.csv,NaN
7,france_politique_2002,https://www.france-politique.fr/sondages-electoraux-pres...,wayback:20260522145707,2026-10-01T09:36:13+00:00,5c712381523b29302213f4f429a117578789597f5715a109e220d9ea...,data/raw/france_politique/sondages-electoraux-presidenti...,https://web.archive.org/web/20260522145707/https://www.f...
8,results,https://raw.githubusercontent.com/clarasalas/strategic-v...,8e99f86b819c889a681a7e5bf30a94e7d67abb7b,2026-10-01T16:07:12+00:00,4f72126b8ab590a3d134b47efdd01aca6de77c63cafefb0cd42f2caa...,data/raw/results/8e99f86b819c_FR-electoral_data.csv,NaN
9,france_politique_2002_wayback,https://web.archive.org/web/20261001093553id_/https://ww...,wayback:20261001163732,2026-10-01T16:39:27+00:00,5c712381523b29302213f4f429a117578789597f5715a109e220d9ea...,data/raw/france_politique/wayback_20261001163732_sondage...,https://web.archive.org/web/20261001163732/https://www.f...


## Part 2 · Parsing (offline from here)

Every cell below reads only from `data/raw/`, using the file locations derived from the pins.

In [4]:
cfg = fetch.load_sources(SOURCES)
paths = fetch.raw_paths(cfg, RAW)
missing = [str(p) for p in [*paths["wikipedia"].values(), paths["france_politique"], paths["nsppolls"], paths["results"]]
           if not p.exists()]
assert not missing, f"Raw files missing; run Part 1 first: {missing}"
errors = []

### Wikipedia (2002–2022)

One parser handles all five pages; their differences are documented in `src/polls/wikipedia.py`.
Only tables under a "premier tour" h2 heading are read. Single dates go to `reported_date`, because the
page doesn't say whether they mark the end of fieldwork or publication. Ranges go to `fieldwork_start`/`fieldwork_end`.

In [5]:
wp_rows = []
for year, path in paths["wikipedia"].items():
    rows, errs = wikipedia.parse_page(path, year)
    wp_rows += rows
    errors += errs
    print(f"{year}: {len(rows):5d} rows, {len({r['page_poll_no'] for r in rows}):4d} polls, {len(errs)} parse errors")

2002:  1537 rows,   98 polls, 0 parse errors
2007:  1290 rows,  107 polls, 0 parse errors
2012:  1913 rows,  124 polls, 0 parse errors
2017:  4885 rows,  154 polls, 1 parse errors
2022:  7349 rows,  263 polls, 0 parse errors


### france-politique (2002)

`ns` → `not_tested`; `autres` → `grouped_others`; blank → `missing`. None of these get a share.
The IFOP tables' "Autres" row is the aggregate share of the grouped candidates and is mapped to the
pseudo-candidate `__others__`. In the CSA and SOFRES/LCI tables, "Noël Mamère*" stands for whichever Green
candidate was tested (Mamère, Voynet or Lipietz), per the page's footnote.

In [6]:
fp_rows, errs = france_politique.parse_page(paths["france_politique"])
errors += errs
print(f"{len(fp_rows)} rows, {len({r['page_poll_no'] for r in fp_rows})} polls, {len(errs)} parse errors")

1746 rows, 98 polls, 0 parse errors


### NSPPolls (2022), first round only

In [7]:
nsp_rows, errs = nsppolls.parse_file(paths["nsppolls"])
errors += errs
print(f"{len(nsp_rows)} rows, {len({r['page_poll_no'] for r in nsp_rows})} polls, {len(errs)} parse errors")

7329 rows, 351 polls, 0 parse errors


### Official first-round results

In [8]:
res = results.parse_file(paths["results"], cfg["results"]["commit"])
res.groupby("election").agg(candidates=("candidate_raw", "size"), total_share=("share", "sum"))

,candidates,total_share
election,,
2002,16,100.02
2007,12,100.01
2012,10,100.00
2017,11,100.00
2022,12,99.98


### Candidate names

`config/candidates.csv` maps `(election, candidate_raw)` to a normalised `candidate`. New raw names are added
on each run and prefilled where the match is unambiguous; hand edits are never overwritten. The cell fails
if any name is left unmapped: fill in the `candidate` column (and set `match` to `manual`) and rerun.

In [9]:
polls_raw = pd.concat([pd.DataFrame(wp_rows), pd.DataFrame(fp_rows), pd.DataFrame(nsp_rows)], ignore_index=True)
mapping = candidates.update_mapping(CANDIDATES, polls_raw, res)
print(mapping["match"].replace("", "(unmapped)").value_counts().to_string())

todo = candidates.unmapped(mapping)
if len(todo):
    raise ValueError(f"{len(todo)} unmapped candidate names; fill them in {CANDIDATES}:\n{todo.to_string()}")

match
results            146
exact               40
surname             37
manual              19
hint                18
initial+surname      2
rule                 1


### Pollster names

`config/institutes.csv` maps each pollster name as written in a source (`institute_raw`) to a normalised
`institute` (the pollster at the time, without partner names such as "-Fiducial" or "Sopra Steria") and an
`institute_family` (the company across renamings: Sofres → TNS Sofres → Kantar; Louis Harris → LH2).
Spelling variants of a known name are filled automatically; anything else must be filled by hand.

In [10]:
inst_map = institutes.update_mapping(INSTITUTES, polls_raw)
print(inst_map["match"].replace("", "(unmapped)").value_counts().to_string())

todo = institutes.unmapped(inst_map)
if len(todo):
    raise ValueError(f"{len(todo)} unmapped pollster names; fill them in {INSTITUTES}:\n{todo.to_string()}")

match
manual      30
spelling     8


### Assemble and write the tables

In [11]:
res_out = candidates.apply_mapping(res, mapping)[assemble.RESULT_COLUMNS]

tables = {}
for source, rows in [("wikipedia", wp_rows), ("france_politique", fp_rows), ("nsppolls", nsp_rows)]:
    df = assemble.to_frame(rows)
    df = assemble.assign_ids(df)
    df = institutes.apply_mapping(df, inst_map)
    df = candidates.apply_mapping(df, mapping)
    df = assemble.flag_final_list(df, res_out)
    tables[source] = assemble.finalise(df)
    tables[source].to_csv(INTERIM / f"polls_{source}.csv", index=False)

res_out.to_csv(INTERIM / "results.csv", index=False)
errors_df = pd.DataFrame(errors, columns=["source", "election", "location", "raw", "reason"])
errors_df.to_csv(INTERIM / "parse_errors.csv", index=False)

for source, df in tables.items():
    assert df["poll_id"].notna().all() and not df.duplicated(["scenario_id", "candidate_raw", "candidate_column"]).any()
    print(f"polls_{source}.csv: {len(df):6d} rows, {df['poll_id'].nunique():4d} polls, "
          f"{df['scenario_id'].nunique():4d} scenarios")
print(f"results.csv: {len(res_out)} rows;  parse_errors.csv: {len(errors_df)} rows")

polls_wikipedia.csv:  16974 rows,  746 polls, 1203 scenarios
polls_france_politique.csv:   1746 rows,   98 polls,   98 scenarios
polls_nsppolls.csv:   7329 rows,  351 polls,  561 scenarios
results.csv: 61 rows;  parse_errors.csv: 1 rows


## Summary report

In [12]:
allpolls = pd.concat(tables.values(), ignore_index=True)
by_election, by_institute = checks.poll_counts(allpolls)
print("Polls per election and source")
by_election

Polls per election and source


source,france_politique,nsppolls,wikipedia
election,,,
2002,98,0,98
2007,0,0,107
2012,0,0,124
2017,0,0,154
2022,0,351,263


In [13]:
print("Polls per election, source and institute")
by_institute.pivot_table(index=["election", "institute"], columns="source", values="polls", fill_value=0)

Polls per election, source and institute


source                                 france_politique  nsppolls  wikipedia
election institute                                                          
2002     BVA                                       10.0       0.0       10.0
         CSA                                       23.0       0.0       23.0
         Ifop                                      16.0       0.0       16.0
         Ipsos                                     20.0       0.0       21.0
         Louis Harris                               5.0       0.0        5.0
         Sofres                                    24.0       0.0       23.0
2007     BVA                                        0.0       0.0       11.0
         CSA                                        0.0       0.0       20.0
         Ifop                                       0.0       0.0       22.0
         Ipsos                                      0.0       0.0       25.0
         LH2                                        0.0       0.0       12.0
         Sofres                                     0.0       0.0       17.0
2012     BVA                                        0.0       0.0       12.0
         CSA                                        0.0       0.0       13.0
         Harris Interactive                         0.0       0.0       12.0
         Ifop                                       0.0       0.0       37.0
         Ipsos                                      0.0       0.0       20.0
         LH2                                        0.0       0.0        9.0
         OpinionWay                                 0.0       0.0       15.0
         Sofres                                     0.0       0.0        6.0
2017     BVA                                        0.0       0.0       20.0
         CSA                                        0.0       0.0        2.0
         Dedicated Research                         0.0       0.0        1.0
         Elabe                                      0.0       0.0       18.0
         Future Thinking - SSI                      0.0       0.0        1.0
         Harris Interactive                         0.0       0.0       11.0
         Ifop                                       0.0       0.0       45.0
         Ipsos                                      0.0       0.0       17.0
         Kantar Sofres                              0.0       0.0        8.0
         Odoxa                                      0.0       0.0       12.0
         OpinionWay                                 0.0       0.0       14.0
         Scan Research - Le Terrain                 0.0       0.0        2.0
         TNS Sofres                                 0.0       0.0        3.0
2022     Atlas Político                             0.0       0.0        1.0
         AtlasIntel                                 0.0       1.0        0.0
         BVA                                        0.0      13.0       13.0
         Cluster17                                  0.0      24.0       17.0
         Elabe                                      0.0      24.0       25.0
         Harris Interactive                         0.0      42.0       44.0
         Ifop                                       0.0      83.0       55.0
         Ipsos                                      0.0      43.0       29.0
         Kantar Public                              0.0       4.0        4.0
         Odoxa                                      0.0       7.0        7.0
         OpinionWay                                 0.0     108.0       66.0
         Redfield & Wilton Strategies               0.0       1.0        0.0
         YouGov                                     0.0       1.0        2.0

In [14]:
sums = checks.share_sums(allpolls)
flagged = sums[sums["flag"]]
print(f"Scenarios with share sum outside {checks.SUM_RANGE}: {len(flagged)} of {len(sums)}")
print("Of these, scenarios with cells that carry no number (<x %, autres, blank):",
      int((flagged[["n_below_threshold", "n_grouped_others", "n_missing"]].sum(axis=1) > 0).sum()))
flagged.groupby(["election", "source"]).size().rename("flagged scenarios").to_frame()

Scenarios with share sum outside (98.0, 102.0): 32 of 1862
Of these, scenarios with cells that carry no number (<x %, autres, blank): 10


,,flagged scenarios
election,source,
2002,wikipedia,5
2007,wikipedia,3
2012,wikipedia,19
2017,wikipedia,1
2022,nsppolls,4


In [15]:
flagged.sort_values("sum_shares")

,election,source,poll_id,scenario_id,institute,sum_shares,n_values,n_below_threshold,n_grouped_others,n_missing,has_others_row,flag
1291,2022,nsppolls,2022_nsp_opinion-way_2022-01-20_2,2022_nsp_opinion-way_2022-01-20_2_s2,OpinionWay,0.0,1,0,0,0,False,True
1293,2022,nsppolls,2022_nsp_opinion-way_2022-01-21_1,2022_nsp_opinion-way_2022-01-21_1_s2,OpinionWay,0.0,1,0,0,0,False,True
1030,2022,nsppolls,2022_nsp_ifop_2020-06-19_1,2022_nsp_ifop_2020-06-19_1_s2,Ifop,2.0,1,0,0,0,False,True
344,2012,wikipedia,2012_wp_harris_2011-12-16_1,2012_wp_harris_2011-12-16_1_s1,Harris Interactive,94.5,8,1,0,0,False,True
1380,2022,nsppolls,2022_nsp_redfield-wilton-strategies_2021-04-15_1,2022_nsp_redfield-wilton-strategies_2021-04-15_1_s1,Redfield & Wilton Strategies,96.0,9,0,0,0,False,True
137,2002,wikipedia,2002_wp_ifop_2002-02-22_1,2002_wp_ifop_2002-02-22_1_s1,Ifop,96.0,13,0,0,0,False,True
231,2007,wikipedia,2007_wp_ifop_2006-11-18_1,2007_wp_ifop_2006-11-18_1_s1,Ifop,96.0,9,0,0,0,False,True
442,2012,wikipedia,2012_wp_lh2_2011-12-11_1,2012_wp_lh2_2011-12-11_1_s1,LH2,96.0,7,2,0,0,False,True
319,2012,wikipedia,2012_wp_csa_2011-12-15_1,2012_wp_csa_2011-12-15_1_s1,CSA,96.0,9,0,0,0,False,True
387,2012,wikipedia,2012_wp_ifop_2012-01-06_1,2012_wp_ifop_2012-01-06_1_s1,Ifop,96.5,7,5,0,0,False,True


In [16]:
dates = checks.date_checks(allpolls)
print(f"Polls with date problems: {len(dates)}")
dates

Polls with date problems: 0


""


In [17]:
samples = checks.sample_checks(allpolls)
print(f"Sample-size flags: {len(samples)}  (plausible range {checks.SAMPLE_RANGE})")
samples

Sample-size flags: 0  (plausible range (300, 30000))


,election,source,poll_id,institute,sample_size,effective_base,issue


In [18]:
print("Share status by source")
allpolls.pivot_table(index="source", columns="share_status", values="poll_id", aggfunc="size", fill_value=0)

Share status by source


share_status,below_threshold,grouped_others,missing,not_tested,value
source,,,,,
france_politique,0,57,6,246,1437
nsppolls,0,0,0,0,7329
wikipedia,447,0,1,2982,13544


In [19]:
print("Scenarios matching the official first-round list (final_list_scenario)")
allpolls.drop_duplicates("scenario_id").pivot_table(
    index="election", columns="source", values="final_list_scenario", aggfunc="sum", fill_value=0)

Scenarios matching the official first-round list (final_list_scenario)


source,france_politique,nsppolls,wikipedia
election,,,
2002,10,0,16
2007,0,0,43
2012,0,0,29
2017,0,0,49
2022,0,135,90


In [20]:
print(f"Parse errors: {len(errors_df)}  (written to data/interim/parse_errors.csv)")
errors_df

Parse errors: 1  (written to data/interim/parse_errors.csv)


,source,election,location,raw,reason
0,wikipedia,2017,table 21 (Sondages réalisés avant la publication de la l...,640-646,not a number: '640-646'
